# Prepare the 32K chat SFT corpus (CPU)

Select **CPU** and turn **Internet on**. Attach the frozen tokenizer dataset
`xiayicheng3gmailcom/nano-dsv41f-tokenizer-fineweb`.

Streams [HuggingFaceTB/smol-smoltalk](https://huggingface.co/datasets/HuggingFaceTB/smol-smoltalk),
the Apache-2.0 SmolTalk subset Hugging Face built for sub-1B models (SmolLM2-135M/360M-Instruct).
It contains no function-calling data. Each conversation is rendered with the
DeepSeek V4.1 encoder in **non-thinking** mode, tokenized with the frozen nano
tokenizer and packed into 32K rows with **assistant-only** loss masks. The dataset's
test split becomes validation, so both SFT arms are scored on the same held-out chats.
The resolved dataset commit is written to `chat_sft_manifest.json`.

Save the output (`chat-sft-corpus/`) and attach it to the TPU notebook. A few
subsets carry upstream terms (OpenHermes / Llama-3.1-generated rows), so keep the
saved corpus private rather than republishing it.

In [ ]:
import os
from pathlib import Path
import subprocess
import sys
os.environ['JAX_PLATFORMS'] = 'cpu'
os.environ['TOKENIZERS_PARALLELISM'] = 'true'
SOURCE_REF = 'main'
MAX_TRAIN_RECORDS = 0  # 0 = whole split
OUTPUT = Path('/kaggle/working/chat-sft-corpus')
ROOT = Path('/kaggle/working/nano-dsv4.1f')
if not ROOT.exists():
    subprocess.run(['git', 'clone', '-q', 'https://github.com/xiayicheng3-code/nano-dsv4.1f.git', str(ROOT)], check=True)
subprocess.run(['git', '-C', str(ROOT), 'fetch', '-q', 'origin', SOURCE_REF], check=True)
subprocess.run(['git', '-C', str(ROOT), 'checkout', '-q', '--detach', 'FETCH_HEAD'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'jax[cpu]==0.10.2', 'tokamax==0.0.12',
                'datasets>=3.0', 'tokenizers>=0.21', 'deepseek-recipe==0.1.1', 'huggingface_hub'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', str(ROOT), '--no-deps'], check=True)
os.chdir(ROOT)
os.environ['PYTHONPATH'] = os.pathsep.join([str(ROOT / 'src'), str(ROOT / 'scripts')])
print('Source commit:', subprocess.check_output(['git', 'rev-parse', 'HEAD'], text=True).strip())

In [ ]:
import json
tokenizers = [p for p in Path('/kaggle/input').rglob('tokenizer.json')
              if 'nano-dsv41f-tokenizer-fineweb' in str(p)]
if len(tokenizers) != 1:
    raise FileNotFoundError('Attach the nano-dsv41f-tokenizer-fineweb dataset')
command = [sys.executable, '-u', 'scripts/prepare_chat_sft_corpus.py', '--output', str(OUTPUT),
           '--tokenizer', str(tokenizers[0]), '--max-train-records', str(MAX_TRAIN_RECORDS)]
subprocess.run(command, check=True)
manifest = json.loads((OUTPUT / 'chat_sft_manifest.json').read_text())
for split, stats in manifest['splits'].items():
    print(split, {k: v for k, v in stats.items() if k not in ('shards', 'subset_tokens')})
    total = sum(stats['subset_tokens'].values())
    for subset, tokens in sorted(stats['subset_tokens'].items(), key=lambda kv: -kv[1]):
        print(f'   {subset:28s} {tokens / total:6.1%}')
print('dataset revision:', manifest['dataset']['resolved_revision'])